# Molecular filtering: ADME and lead-likeness criteria

## Aim of this talktorial:
In the context of drug design, it is important to filter candidate molecules by e.g. their physicochemical properties. Here, the compounds acquired from ChEMBL will be filtered by Lipinsik's rule of five to keep only orally bioavailable compounds.

## For more details
https://github.com/volkamerlab/teachopencadd/blob/master/teachopencadd/talktorials/T002_compound_adme/talktorial.ipynb

## Instructions
Replace XXX with the appropriate code

## Configuration

In [ ]:
# Imports
# 1. Standard library imports
import math
from pathlib import Path
import sys
sys.path.append('../my_modules') # to tell where to find local modules

# 2. Third-party library imports
import matplotlib.patches as mpatches
import matplotlib.pyplot as plt
from matplotlib.lines import Line2D
import numpy as np
import pandas as pd
from rdkit import Chem
from rdkit.Chem import (
    Descriptors,
    Draw,
    PandasTools
)
PandasTools.RenderImagesInAllDataFrames(images=True) # to molecules as images in DataFrames
from rdkit.Chem.Draw import IPythonConsole # needed to show molecules
# # from rdkit.Chem.Draw.MolDrawing import MolDrawing, DrawingOptions # only needed if modifying defaults

# 3. Local application imports
import kernel_infos

In [ ]:
# Information about the kernel
kernel_infos.show_kernel_info()

In [ ]:
# Global variables
HERE = Path().resolve()
print(f'{HERE}')
ROOT = HERE.parent
print(f'{ROOT}')
DATA = ROOT / 'data'
print(f'{DATA}')

## Exemple molecules 

### Build a daframe with molecules

In [ ]:
# Lists of smiles and names
smiles_l = [
    "CCC1C(=O)N(CC(=O)N(C(C(=O)NC(C(=O)N(C(C(=O)NC(C(=O)NC(C(=O)N(C(C(=O)N(C(C(=O)N(C(C(=O)N(C(C(=O)N1)C(C(C)CC=CC)O)C)C(C)C)C)CC(C)C)C)CC(C)C)C)C)C)CC(C)C)C)C(C)C)CC(C)C)C)C",
    "CN1CCN(CC1)C2=C3C=CC=CC3=NC4=C(N2)C=C(C=C4)C",
    "CC1=C(C(CCC1)(C)C)C=CC(=CC=CC(=CC=CC=C(C)C=CC=C(C)C=CC2=C(CCCC2(C)C)C)C)C",
    "CCCCCC1=CC(=C(C(=C1)O)C2C=C(CCC2C(=C)C)C)O",
]
names_l = ["cyclosporine", "clozapine", "beta-carotene", "cannabidiol"]

In [ ]:
# Dataframe with previous data
mols_df = pd.DataFrame({"name":XXX, "smiles":XXX})

In [ ]:
print(f"Dataframe shape: {mols_df.XXX}")
mols_df

In [ ]:
# Add a molecule column
PandasTools.XXX(mols_df, 'smiles', 'ROMol')

In [ ]:
print(f"Dataframe shape: {mols_df.shape}")
mols_df

### Draw an image of compounds with their name

In [ ]:
img = Draw.XXX(
    mols_df['ROMol'].tolist(),
    legends=XXX.tolist(),
    molsPerRow=4,
    subImgSize=(300,300))
img

### Compute molecular properties defining ro5

- MW: molecular_weight
- HBA: number of hydrogen bond acceptors
- HBD: number of hydrogen bond donors
- logP: octanol-water partition coefficient


In [ ]:
# Property list
properties = ['MW', 'HBA', 'HBD', 'LogP']

# RDKit functions to compute properties
property_functions = {
    'MW': Descriptors.ExactMolWt,
    'HBA': Descriptors.XXX,
    'HBD': Descriptors.XXX,
    'LogP': Descriptors.MolLogP 
}

In [ ]:
# Complete mols_df with properties defined from ROMol's molecule
for prop in properties:
    func = property_functions[prop]
    mols_df[prop] = mols_df[XXX].apply(lambda mol: func(mol))

print(f"Dataframe shape: {mols_df.shape}")
mols_df.head(2) 

In [ ]:
# Add a color column: one molecule <-> one color, from a color map (modulo the number of colors in the cmap)

cmap = plt.get_cmap('tab10') # select a specify cmap, tab10 : 10 separated colors
mols_df['color'] = [cmap(iXXXlen(cmap.colors)) for i in range(mols_df.shape[0])]
mols_df.head(2)

### Plot molecular propertie distributions as bar plots

In [ ]:
# Dictionnary of lead-likeness criteria
# key: (threshold, title) for the plots
ro5_thresholds = {
    'MW': (500, "molecular weight (Da)"),
    'HBA': (10, "# of H-bond acceptors"),
    'HBD': (5, "# of H-bond donors"),
    'LogP': (5, "LogP")
}

In [ ]:
# Print dictionnary items
print(f"Dictionnary items:")
for key, value in ro5_thresholds.XXX:
    print(f"Key: {key}, Value: {value}")

In [ ]:
fig, axes = plt.subplots(figsize=(10, 2.5), nrows=1, ncols=4)
# 1 subplot by property, 1 bar by molecule
molecule_index = np.arange(1, len(mols_df)+1) # molecule indexes, from 1 to number of molecules

colors = mols_df['color'].tolist() # colors list

# Create subplots
for property_index, (key, (threshold, title)) in enumerate(ro5_thresholds.items()):
    
    axes[property_index].bar(molecule_index, mols_df[key], color = colors) # bar plot
    axes[property_index].axhline(
        y=threshold,
        color='black',
        linestyle='--',
    ) # threshold line
    axes[property_index].set_title(f'{key} distribution') # title (replaced by key because of length issues)
    axes[property_index].set_xticks([]) # remove x_ticks

# Add legend for each molecule by iterating over mols_df rows
legend_elements = [
    mpatches.Patch(
        color = row['color'],
        label = row['name']
    ) for _, row in mols_df.iterrows()
]
# Add threshold line to legend
legend_elements.append(Line2D([0], [0], color="black", ls="dashed", label="Threshold"))
fig.legend(handles=legend_elements, bbox_to_anchor=(1.2, 0.8))

# Fit subplots and legend properly
fig.tight_layout()
plt.show()



### Investigate compliance to Lipinski's rule of five

In [ ]:
# Create a function to test if an input molecule (smiles) fulfills Lipinski's rule of five
def calculate_ro5_properties(smiles):
    # RDKit molecule from smiles
    mol = Chem.MolFromSmiles(XXX)
    # Compute ro5 properties
    MW = XXX.ExactMolWt(mol)
    HBA = XXX.NumHAcceptors(mol)
    HBD = XXX.NumHDonors(mol)
    LogP = XXX.MolLogP(mol)
    # Check if ro5 conditions are met
    conditions = [MW <= 500, HBA <= 10, HBD <= 5, LogP <= 5]
    ro5_fulfilled = XXX(conditions) >= 3
    # Return the molecular properties and True if ro5 is fulfilled, False otherwise
    return pd.Series(
        [MW, HBA, HBD, LogP, ro5_fulfilled],
        index=['MW', 'HBA', 'HBD', 'LogP', 'ro5_fulfilled']
    )

In [ ]:
# Test is on example molecules
for name, smiles in XXX(names_l, smiles_l):
    print(f"ro5 fullfilled for {name}: {calculate_ro5_properties(smiles)['ro5_fulfilled']}")

## Apply ro5 to EGFR dataset

In [ ]:
# Load data from the previous notebook in a dataframe
csv_file_path = DATA / "EGFR_CHEMBL36_output.csv"
mols_df = pd.XXX(csv_file_path, index_col=0)
print(f"Dataframe shape: {mols_df.shape}")
mols_df.head(2)

In [ ]:
# Apply calculate_ro5_properties to the dataframe (could take quite a long time)
ro5properties = mols_df['smiles'].apply(XXX)

In [ ]:
print(f"Dataframe shape: {ro5properties.shape}")
ro5properties.head(2)

In [ ]:
# Concatenate molecules and ro5 properties
mols_df = pd.XXX([mols_df, ro5properties], axis=1)
print(f"Dataframe shape: {mols_df.shape}")
mols_df.head(2)

In [ ]:
# Explore column ro5_fulfilled
print(mols_df['XXX'].unique())
mols_df.value_counts(subset='XXX')

In [ ]:
# Separate the two types of compounds regarding ro5_fulfilled
mols_ro5_fulfilled_df = mols_df[mols_df['ro5_fulfilled']]
mols_ro5_violated_df = mols_df[~mols_df['ro5_fulfilled']]

In [ ]:
print(f"Number of compounds in unfiltered data: {XXX.shape[0]}")
print(f"Number of compounds compliant with ro5: {XXX.shape[0]}")
print(f"Number of compounds not compliant with ro5: {XXX.shape[0]}")

In [ ]:
# Save ro5 compliant data
csv_file_path = DATA / "EGFR_compounds_lipinski.csv"
mols_ro5_fulfilled_df.XXX(csv_file_path)

## Visualise ro5 properties within a radar plot

### Compute some statistics: mean and standard deviation

In [ ]:
# from df.describe
def calculate_mean_std(dataframe):
    """
    Calculate the mean and standard deviation of a dataset.

    Parameters
    ----------
    dataframe : pd.DataFrame
        Properties (columns) for a set of items (rows).

    Returns
    -------
    pd.DataFrame
        Mean and standard deviation (columns) for different properties (rows).
    """
    # Generate descriptive statistics for property columns
    stats = dataframe.describe()
    # Transpose DataFrame (statistical measures = columns)
    stats = stats.T
    # Select mean and standard deviation
    stats = stats[["mean", "std"]]
    return stats

In [ ]:
# # from std, mean
# def calculate_stats(properties_df):
#     mean_l = list()
#     std_l = list()

#     for prop in properties_df.columns:
#         mean_l.append(properties_df[prop].mean())
#         std_l.append(properties_df[prop].std())

#     df = pd.DataFrame({
#         'mean': mean_l,
#         'std': std_l
#     })

#     df.index = properties_df.columns

#     return df

### ro5 fulfilled compounds

In [ ]:
# Compute a dataframe with help of the previous function
mols_ro5_fulfilled_stats_df = XXX(
    mols_ro5_fulfilled_df[["MW", "HBA", "HBD", "LogP"]]
)
mols_ro5_fulfilled_stats_df

### ro5 violated compounds

In [ ]:
# Compute a dataframe with help of the previous function
mols_ro5_violated_stats_df = XXX(
    mols_ro5_violated_df[["MW", "HBA", "HBD", "LogP"]]
)
mols_ro5_violated_stats_df

## Helper functions to prepare data for radar plotting

Properties have different magnitudes, so data must be scaled : 
- MW_threshold = 500
- HBA_threshold = 10
- HBD_threshold = 5
- LogP_threshold = 5

In [ ]:
def _scale_by_thresholds(stats_df, thresholds_d, scaled_threshold):
    """
    Scale values for different properties that have each an individually defined threshold.

    Parameters
    ----------
    stats_df : pd.DataFrame
        Dataframe with "mean" and "std" (columns) for each physicochemical property (rows).
    thresholds : dict of str: int
        Thresholds defined for each property.
    scaled_threshold : int or float
        Scaled thresholds across all properties.

    Returns
    -------
    pd.DataFrame
        DataFrame with scaled means and standard deviations for each physiochemical property.
    """

    # Scale property data
    stats_scaled_df = stats_df.apply(lambda x: x / thresholds_d[x.name] * scaled_threshold, axis=1)
    return stats_scaled_df

In [ ]:
def _define_radial_axes_angles(n_axes):
    """Define angles (radians) for radial (x-)axes depending on the number of axes."""
    x_angles = [i / float(n_axes) * 2 * math.pi for i in range(n_axes)]
    x_angles.append(x_angles[0]) # to close lines
    return x_angles

## Generate radar plots

In [ ]:
def plot_radar(
    y,
    thresholds,
    scaled_threshold,
    properties_labels,
    y_max=None,
    output_path=None,
):
    """
    Plot a radar chart based on the mean and standard deviation of a data set's properties.

    Parameters
    ----------
    y : pd.DataFrame
        Dataframe with "mean" and "std" (columns) for each physicochemical property (rows).
    thresholds : dict of str: int
        Thresholds defined for each property.
    scaled_threshold : int or float
        Scaled thresholds across all properties.
    properties_labels : list of str
        List of property names to be used as labels in the plot.
    y_max : None or int or float
        Set maximum y value. If None, let matplotlib decide.
    output_path : None or pathlib.Path
        If not None, save plot to file.
    """

    # Define radial x-axes angles -- uses our helper function!
    x = _define_radial_axes_angles(len(y))
    # Scale y-axis values with respect to a defined threshold -- uses our helper function!
    y = _scale_by_thresholds(y, thresholds, scaled_threshold)
    # Since our chart will be circular we append the first value of each property to the end
    y = pd.concat([y, y.head(1)])

    # Set figure and subplot axis
    plt.figure(figsize=(6, 6))
    ax = plt.subplot(111, polar=True)

    # Plot data
    ax.fill(x, [scaled_threshold] * len(x), "cornflowerblue", alpha=0.2)
    ax.plot(x, y["mean"], "b", lw=3, ls="-")
    ax.plot(x, y["mean"] + y["std"], "orange", lw=2, ls="--")
    ax.plot(x, y["mean"] - y["std"], "orange", lw=2, ls="-.")

    # From here on, we only do plot cosmetics
    # Set 0° to 12 o'clock
    ax.set_theta_offset(math.pi / 2)
    # Set clockwise rotation
    ax.set_theta_direction(-1)

    # Set y-labels next to 180° radius axis
    ax.set_rlabel_position(180)
    # Set number of radial axes' ticks and remove labels
    plt.xticks(x, [])
    # Get maximal y-ticks value
    if not y_max:
        y_max = int(ax.get_yticks()[-1])
    # Set axes limits
    plt.ylim(0, y_max)
    # Set number and labels of y axis ticks
    plt.yticks(
        range(1, y_max),
        ["5" if i == scaled_threshold else "" for i in range(1, y_max)],
        fontsize=16,
    )

    # Draw ytick labels to make sure they fit properly
    # Note that we use [:1] to exclude the last element which equals the first element (not needed here)
    for i, (angle, label) in enumerate(zip(x[:-1], properties_labels)):
        if angle == 0:
            ha = "center"
        elif 0 < angle < math.pi:
            ha = "left"
        elif angle == math.pi:
            ha = "center"
        else:
            ha = "right"
        ax.text(
            x=angle,
            y=y_max + 1,
            s=label,
            size=16,
            horizontalalignment=ha,
            verticalalignment="center",
        )

    # Add legend relative to top-left plot
    labels = ("rule of five area", "mean", "mean + std", "mean - std")
    ax.legend(labels, loc=(1.1, .7), labelspacing=0.3, fontsize=16)

    # Save plot - use bbox_inches to include text boxes
    if output_path:
        plt.savefig(output_path, dpi=300, bbox_inches="tight", transparent=True)

    plt.show()

In [ ]:
# Input parameters
thresholds = {"MW": 500, "HBA": 10, "HBD": 5, "LogP": 5}
scaled_threshold = 5
properties_labels = [
    "MW (Da) / 100",
    "# HBA / 2",
    "# HBD",
    "LogP",
]
y_max = 8

### Radar plot for compounds that fulfill ro5

In [ ]:
plot_radar(
    XXX,
    thresholds,
    scaled_threshold,
    properties_labels,
    y_max,
)

### Radar plot for compounds that violate ro5

In [ ]:
plot_radar(
    XXX,
    thresholds,
    scaled_threshold,
    properties_labels,
    y_max,
)